# AlphaZero Training — Time Analysis (Connect4)

This notebook profiles the main components of the AlphaZero training pipeline to identify bottlenecks.

**Structure:**
1. Setup — load config, build net and env
2. Micro-benchmarks — individual operations in isolation
3. MCTS phases — selection / expansion / evaluation / backprop
4. cProfile — single `mcts.run()` call
5. **Batched MCTS** — `run()` vs `run_batched()` speedup across batch sizes
6. Self-play game — end-to-end timing for sequential vs batched
7. Training step — batch construction + optimizer step
8. Evaluation — `test_vs_mcts` sample
9. Iteration projection — extrapolate to full training iteration (sequential vs batched)
10. Summary chart

In [ ]:
%matplotlib inline

import cProfile
import io
import pstats
import sys
import time
import timeit
from contextlib import contextmanager
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import torch
import yaml

# Add project root to path if needed
ROOT = Path("../..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Import train after matplotlib is already configured — train.py calls
# matplotlib.use("Agg") at module level, but that is a no-op once pyplot
# has already been imported with the inline backend active.
from giotto.agents.algorithms.alphazero.mcts import AlphaZeroMCTS
from giotto.agents.algorithms.alphazero.net import AlphaZeroNet
from giotto.agents.alphazero import AlphaZeroAgent
from giotto.agents.mcts import MCTSAgent
from giotto.envs.connect4 import Connect4Env
from giotto.utils.text_play import play_n_games

print("PyTorch version:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())

## 1. Setup

In [ ]:
CONFIG_PATH = ROOT / "giotto/agents/algorithms/alphazero/config_c4.yaml"
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

print("Config loaded:")
print(f"  Network: {config['network']['residual_blocks']} res-blocks, {config['network']['channels']} channels")
print(f"  MCTS: {config['mcts']['n_sims']} simulations, cpuct={config['mcts']['cpuct']}")
print(f"  Training: {config['training']['games_per_iteration']} games/iter, batch={config['training']['batch_size']}")

env = Connect4Env()

net = AlphaZeroNet(
    input_size=config["network"]["input_size"],
    value_output_size=config["network"]["value_output_size"],
    policy_output_size=config["network"]["policy_output_size"],
    channels=config["network"]["channels"],
    residual_blocks=config["network"]["residual_blocks"],
)
net.eval()

mcts = AlphaZeroMCTS(
    net=net,
    n_simulations=config["mcts"]["n_sims"],
    cpuct=config["mcts"]["cpuct"],
    dirichlet_alpha=config["mcts"]["dirichlet_alpha"],
    dirichlet_eps=config["mcts"]["dirichlet_epsilon"],
)

N_SIMS = config["mcts"]["n_sims"]
print(f"\nSetup complete. N_SIMS={N_SIMS}")

## 2. Micro-benchmarks

Timing individual operations in isolation to get baseline costs.

In [ ]:
REPEATS = 1000
micro = {}

# ---- env.clone() ----
t = timeit.timeit(lambda: env.clone(), number=REPEATS)
micro["env.clone()"] = t / REPEATS

# ---- env.get_valid_actions() ----
t = timeit.timeit(lambda: env.get_valid_actions(), number=REPEATS)
micro["env.get_valid_actions()"] = t / REPEATS

# ---- env.check_win() ----
t = timeit.timeit(lambda: env.check_win(0), number=REPEATS)
micro["env.check_win()"] = t / REPEATS


# ---- env.step() ----
def _step_reset():
    e = env.clone()
    e.step(4)


t = timeit.timeit(_step_reset, number=REPEATS)
micro["env.step()"] = t / REPEATS

# ---- net.predict() ----
state = env.get_state()
t = timeit.timeit(lambda: net.predict(state), number=REPEATS)
micro["net.predict()"] = t / REPEATS

# ---- net.batch_predict() x32 ----
states_batch = [env.get_state() for _ in range(32)]
t = timeit.timeit(lambda: net.batch_predict(states_batch), number=100)
micro["net.batch_predict(x32)"] = (t / 100) / 32  # per-state cost

# ---- AZNode.select_child() ----
mcts.reset()
root = mcts._build_root(env)
# Run a few sims to populate children with visits
for _ in range(50):
    node = root
    while not node.is_terminal() and not node.is_leaf():
        node = node.select_child(config["mcts"]["cpuct"])
    node.expand()
    _, v = net.predict(node.env.get_state())
    node.backpropagate(float(v))

t = timeit.timeit(lambda: root.select_child(config["mcts"]["cpuct"]), number=REPEATS)
micro["node.select_child()"] = t / REPEATS

# ---- AZNode.backpropagate() ----
child = next(iter(root.children.values()))
t = timeit.timeit(lambda: child.backpropagate(0.5), number=REPEATS)
micro["node.backpropagate()"] = t / REPEATS

print("Micro-benchmark results (µs per call):")
print("-" * 45)
for name, seconds in sorted(micro.items(), key=lambda x: -x[1]):
    print(f"  {name:<30}  {seconds * 1e6:8.2f} µs")

## 3. MCTS phases breakdown

Time each phase of a simulation (selection, expansion, evaluation, backprop) for a mid-game state.

In [ ]:
# Create a mid-game state (5 random moves played)
mid_env = Connect4Env()
mid_env.reset()
for action in [4, 4, 3, 5, 2]:
    mid_env.step(action)

phase_times = {"selection": 0.0, "expansion": 0.0, "evaluation": 0.0, "backprop": 0.0}

# Warm-up root
mcts.reset()
mcts_root = mcts._build_root(mid_env)

N_PROFILE_SIMS = N_SIMS

for _ in range(N_PROFILE_SIMS):
    node = mcts_root

    # SELECTION
    t0 = time.perf_counter()
    while not node.is_terminal() and not node.is_leaf():
        node = node.select_child(config["mcts"]["cpuct"])
    phase_times["selection"] += time.perf_counter() - t0

    if not node.is_terminal():
        # EXPANSION
        t0 = time.perf_counter()
        node.expand()
        phase_times["expansion"] += time.perf_counter() - t0

        # EVALUATION
        t0 = time.perf_counter()
        policy, value = net.predict(node.env.get_state())
        valid_actions = node.env.get_valid_actions()
        raw = np.array([policy[a - 1] for a in valid_actions], dtype=np.float32)
        raw /= raw.sum()
        for action, prob in zip(valid_actions, raw):
            node.children[action].prob = float(prob)
        value = float(value)
        phase_times["evaluation"] += time.perf_counter() - t0
    else:
        value = node.terminal_node_eval(node.env, node.to_play)

    # BACKPROPAGATION
    t0 = time.perf_counter()
    node.backpropagate(value)
    phase_times["backprop"] += time.perf_counter() - t0

total_mcts_time = sum(phase_times.values())
print(f"MCTS phases over {N_PROFILE_SIMS} simulations (total {total_mcts_time:.3f}s):")
print("-" * 55)
for phase, t in phase_times.items():
    pct = 100.0 * t / total_mcts_time
    per_sim = t / N_PROFILE_SIMS * 1000
    print(f"  {phase:<14} {t:6.3f}s  ({pct:5.1f}%)  {per_sim:.3f} ms/sim")

## 4. cProfile — single MCTS run

Full `cProfile` on a single `mcts.run()` call (using config n_sims).

In [ ]:
@contextmanager
def profile_block(sort="cumtime", limit=30):
    pr = cProfile.Profile()
    pr.enable()
    try:
        yield
    finally:
        pr.disable()
        s = io.StringIO()
        pstats.Stats(pr, stream=s).strip_dirs().sort_stats(sort).print_stats(limit)
        print(s.getvalue())


fresh_env = Connect4Env()
fresh_env.reset()
mcts.reset()

with profile_block(sort="tottime", limit=25):
    action, root = mcts.run(fresh_env, temperature=0.0)

## 5. Batched MCTS — `run()` vs `run_batched()` speedup

Sweep batch sizes to find the optimal leaf-parallelism factor. Each measurement
resets the tree so no warm-cache advantage is given to later runs.

In [ ]:
BATCH_SIZES_TO_TEST = [1, 2, 4, 8, 16, 32]
N_BENCH_REPEATS = 3  # repeat each measurement and take the mean

# Use a mid-game board so the tree has realistic depth
bench_env = Connect4Env()
bench_env.reset()
for _action in [4, 3, 4, 3, 2, 5]:
    bench_env.step(_action)

batch_bench: dict[int, dict] = {}

# ── Baseline: sequential run() ────────────────────────────────────────────
seq_times = []
for _ in range(N_BENCH_REPEATS):
    m = AlphaZeroMCTS(
        net=net,
        n_simulations=N_SIMS,
        cpuct=config["mcts"]["cpuct"],
        dirichlet_alpha=config["mcts"]["dirichlet_alpha"],
        dirichlet_eps=config["mcts"]["dirichlet_epsilon"],
        batch_size=1,
    )
    t0 = time.perf_counter()
    m.run(bench_env, temperature=0.0)
    seq_times.append(time.perf_counter() - t0)

baseline_batch_time = float(np.mean(seq_times))
print(f"{'run()  [sequential]':<32}  {baseline_batch_time:.3f}s   speedup: 1.00×")

# ── Batched: run_batched() across batch sizes ─────────────────────────────
for bs in BATCH_SIZES_TO_TEST:
    bt = []
    for _ in range(N_BENCH_REPEATS):
        m = AlphaZeroMCTS(
            net=net,
            n_simulations=N_SIMS,
            cpuct=config["mcts"]["cpuct"],
            dirichlet_alpha=config["mcts"]["dirichlet_alpha"],
            dirichlet_eps=config["mcts"]["dirichlet_epsilon"],
            batch_size=bs,
        )
        t0 = time.perf_counter()
        m.run_batched(bench_env, temperature=0.0)
        bt.append(time.perf_counter() - t0)
    avg_t = float(np.mean(bt))
    speedup = baseline_batch_time / avg_t
    batch_bench[bs] = {"time": avg_t, "speedup": speedup}
    label = f"run_batched(batch={bs})"
    print(f"  {label:<30}  {avg_t:.3f}s   speedup: {speedup:.2f}×")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle(f"run() vs run_batched()  —  {N_SIMS} simulations, Connect4", fontsize=12, fontweight="bold")

bs_list = list(batch_bench.keys())
times_list = [batch_bench[bs]["time"] for bs in bs_list]
speedups = [batch_bench[bs]["speedup"] for bs in bs_list]
config_bs = config["mcts"].get("batch_size", 8)

# ── Chart 1: absolute time ────────────────────────────────────────────────
ax = axes[0]
ax.axhline(baseline_batch_time, color="tomato", linewidth=2, linestyle="--", label="run()  sequential")
ax.plot(bs_list, times_list, "o-", color="steelblue", linewidth=2, markersize=7, label="run_batched()")
ax.axvline(config_bs, color="grey", linewidth=1, linestyle=":", label=f"config batch_size={config_bs}")
ax.set_xlabel("Batch size")
ax.set_ylabel("Time (s)")
ax.set_title("MCTS wall time per search")
ax.set_xticks(bs_list)
ax.legend()
ax.grid(True, alpha=0.3)
for x, y in zip(bs_list, times_list):
    ax.annotate(f"{y:.2f}s", (x, y), textcoords="offset points", xytext=(4, 6), fontsize=8)

# ── Chart 2: speedup ─────────────────────────────────────────────────────
ax = axes[1]
ax.axhline(1.0, color="tomato", linewidth=2, linestyle="--", label="run()  baseline  (1×)")
ax.plot(bs_list, speedups, "o-", color="steelblue", linewidth=2, markersize=7, label="run_batched() speedup")
ax.axvline(config_bs, color="grey", linewidth=1, linestyle=":", label=f"config batch_size={config_bs}")
ax.set_xlabel("Batch size")
ax.set_ylabel("Speedup  (×)")
ax.set_title("Speedup over sequential run()")
ax.set_xticks(bs_list)
ax.legend()
ax.grid(True, alpha=0.3)
for x, y in zip(bs_list, speedups):
    ax.annotate(f"{y:.2f}×", (x, y), textcoords="offset points", xytext=(4, 6), fontsize=8)

plt.tight_layout()
plt.show()

best_bs = max(batch_bench, key=lambda k: batch_bench[k]["speedup"])
print(f"\nBest batch size: {best_bs}  →  {batch_bench[best_bs]['speedup']:.2f}× speedup over sequential")

## 5. Self-play game timing

Time a complete self-play game and record per-move breakdown.

In [ ]:
N_SAMPLE_GAMES = 3  # small sample for speed
GAME_BATCH_SIZE = config["mcts"].get("batch_size", 8)

net.eval()


def _time_games(use_batched: bool, batch_size: int = 1) -> tuple[list[float], list[int]]:
    """Play N_SAMPLE_GAMES and return (game_times, move_counts)."""
    g_times, m_counts = [], []
    for g in range(N_SAMPLE_GAMES):
        m = AlphaZeroMCTS(
            net=net,
            n_simulations=N_SIMS,
            cpuct=config["mcts"]["cpuct"],
            dirichlet_alpha=config["mcts"]["dirichlet_alpha"],
            dirichlet_eps=config["mcts"]["dirichlet_epsilon"],
            batch_size=batch_size,
        )
        game_env = Connect4Env()
        game_env.reset()
        move_count = 0
        t_start = time.perf_counter()
        while not game_env.done:
            schedule = config["mcts"]["temperature_schedule"]
            temperature = config["mcts"]["temperature"] if move_count < schedule else 0.0
            if use_batched:
                action, _ = m.run_batched(game_env, temperature=temperature)
            else:
                action, _ = m.run(game_env, temperature=temperature)
            game_env.step(action)
            m.advance_root(action)
            move_count += 1
        g_times.append(time.perf_counter() - t_start)
        m_counts.append(move_count)
    return g_times, m_counts


# ── Sequential run() ──────────────────────────────────────────────────────
print("── run()  [sequential] ──")
game_times, move_counts = _time_games(use_batched=False)
avg_game_time = float(np.mean(game_times))
avg_moves = float(np.mean(move_counts))
avg_ms_per_move = avg_game_time / avg_moves * 1000
for g, (t, mc) in enumerate(zip(game_times, move_counts)):
    print(f"  Game {g + 1}: {mc} moves  {t:.2f}s  ({t / mc * 1000:.0f} ms/move)")
print(f"  Average: {avg_game_time:.2f}s/game  {avg_moves:.1f} moves/game  {avg_ms_per_move:.0f} ms/move\n")

# ── Batched run_batched() ─────────────────────────────────────────────────
print(f"── run_batched(batch={GAME_BATCH_SIZE}) ──")
game_times_b, move_counts_b = _time_games(use_batched=True, batch_size=GAME_BATCH_SIZE)
avg_game_time_b = float(np.mean(game_times_b))
avg_moves_b = float(np.mean(move_counts_b))
avg_ms_per_move_b = avg_game_time_b / avg_moves_b * 1000
for g, (t, mc) in enumerate(zip(game_times_b, move_counts_b)):
    print(f"  Game {g + 1}: {mc} moves  {t:.2f}s  ({t / mc * 1000:.0f} ms/move)")
print(f"  Average: {avg_game_time_b:.2f}s/game  {avg_moves_b:.1f} moves/game  {avg_ms_per_move_b:.0f} ms/move")

game_speedup = avg_game_time / avg_game_time_b
print(f"\nSelf-play speedup (batch={GAME_BATCH_SIZE} vs sequential): {game_speedup:.2f}×")

## 6. Training step timing

In [ ]:
import copy
import random

import torch.nn.functional as F

BATCH_SIZE = config["training"]["batch_size"]


# Generate synthetic records with realistic board states (random pieces placed)
def make_fake_records(n):
    records = []
    rng = np.random.default_rng(42)
    for _ in range(n):
        board = np.full((6, 7), -1, dtype=np.int64)
        n_pieces = rng.integers(0, 20)
        for _ in range(n_pieces):
            col = rng.integers(0, 7)
            empty_rows = np.where(board[:, col] == -1)[0]
            if len(empty_rows) > 0:
                board[empty_rows[0], col] = rng.integers(0, 2)
        policy = rng.dirichlet(np.ones(7)).astype(np.float32)
        records.append(
            {
                "state": [board.copy(), int(rng.integers(0, 2))],
                "policy_target": policy,
                "value_target": float(rng.choice([-1.0, 0.0, 1.0])),
            }
        )
    return records


fake_buffer = make_fake_records(2000)

# Train a deep copy so the original net's BatchNorm running stats are not corrupted
net_train = copy.deepcopy(net)
net_train.train()
optimizer = torch.optim.AdamW(net_train.parameters(), lr=config["training"]["learning_rate"])


def train_step(batch_records):
    boards = np.stack([r["state"][0] for r in batch_records])
    players = np.array([r["state"][1] for r in batch_records])
    current = (boards == players[:, None, None]).astype(np.float32)
    opponent = (boards == (1 - players)[:, None, None]).astype(np.float32)
    x = np.stack([current, opponent], axis=1)
    states = torch.from_numpy(x)
    policy_targets = torch.from_numpy(np.stack([r["policy_target"] for r in batch_records]))
    value_targets = torch.tensor([r["value_target"] for r in batch_records], dtype=torch.float32).unsqueeze(1)

    policy_logits, value_preds = net_train(states)
    log_policy = F.log_softmax(policy_logits, dim=1)
    policy_loss = -(policy_targets * log_policy).sum(dim=1).mean()
    value_loss = F.mse_loss(value_preds, value_targets)
    total_loss = policy_loss + value_loss

    optimizer.zero_grad(set_to_none=True)
    total_loss.backward()
    optimizer.step()
    return float(total_loss.item())


# Warm-up
for _ in range(3):
    train_step(random.sample(fake_buffer, BATCH_SIZE))

# Timed run
N_STEPS = 50
step_times = []
for _ in range(N_STEPS):
    batch = random.sample(fake_buffer, BATCH_SIZE)
    t0 = time.perf_counter()
    train_step(batch)
    step_times.append(time.perf_counter() - t0)

avg_step = np.mean(step_times)
print(f"Training step (batch={BATCH_SIZE}):")
print(f"  avg = {avg_step*1000:.1f} ms")
print(f"  p50 = {np.percentile(step_times, 50)*1000:.1f} ms")
print(f"  p95 = {np.percentile(step_times, 95)*1000:.1f} ms")

## 7. Evaluation — `test_vs_mcts` sample

Time a small number of evaluation games to extrapolate to the configured `n_matches`.

In [ ]:
N_EVAL_SAMPLE = 2  # small sample — extrapolate to full
N_MATCHES_FULL = int(config["eval"]["n_matches"])

# Use a fresh net so this cell is independent of training-step side effects
# (training on all-zero boards corrupts BatchNorm running stats, causing NaN probs)
net_eval = AlphaZeroNet(
    input_size=config["network"]["input_size"],
    value_output_size=config["network"]["value_output_size"],
    policy_output_size=config["network"]["policy_output_size"],
    channels=config["network"]["channels"],
    residual_blocks=config["network"]["residual_blocks"],
)
net_eval.eval()

az_agent = AlphaZeroAgent(
    game=config["game"],
    simulations=config["mcts"]["n_sims"],
    cpuct=config["mcts"]["cpuct"],
    net=net_eval,
)
mcts_agent = MCTSAgent(
    simulations=config["mcts"]["n_sims"],
    cpuct=config["mcts"]["cpuct"],
)
agents = [az_agent, mcts_agent]

t0 = time.perf_counter()
play_n_games(N_EVAL_SAMPLE, env.clone(), agents, invert_starts=True)
eval_sample_time = time.perf_counter() - t0

eval_per_game = eval_sample_time / N_EVAL_SAMPLE
eval_full_est = eval_per_game * N_MATCHES_FULL

print(f"Evaluation ({N_EVAL_SAMPLE} games): {eval_sample_time:.2f}s  →  {eval_per_game:.2f}s/game")
print(f"Estimated for {N_MATCHES_FULL} matches (full eval): {eval_full_est:.1f}s  ({eval_full_est/60:.1f} min)")

## 8. Iteration time projection

Extrapolate measured times to a full training iteration.

In [ ]:
GAMES_PER_ITER = int(config["training"]["games_per_iteration"])
BUFFER_SIZE = int(config["training"]["buffer_size"])
training_steps_per_iter = int(config["training"].get("training_steps_per_iteration", 0))

# Estimate steps: if not set, trainer uses len(buffer) // batch_size (capped at buffer_size)
if training_steps_per_iter <= 0:
    est_steps = min(BUFFER_SIZE, GAMES_PER_ITER * int(avg_moves) * 2) // BATCH_SIZE
else:
    est_steps = training_steps_per_iter

t_selfplay_seq = avg_game_time * GAMES_PER_ITER
t_selfplay_bat = avg_game_time_b * GAMES_PER_ITER
t_training = avg_step * est_steps
t_eval = eval_full_est if config["eval"]["play_vs_mcts"] else 0.0

projection_seq = {"Self-play (sequential)": t_selfplay_seq, "Training steps": t_training, "Eval vs MCTS": t_eval}
projection_bat = {"Self-play (batched)": t_selfplay_bat, "Training steps": t_training, "Eval vs MCTS": t_eval}
t_total_seq = sum(projection_seq.values())
t_total_bat = sum(projection_bat.values())

print("Projected time per iteration (single process):")
print(f"{'':30}  {'Sequential':>12}  {'Batched (×' + str(GAME_BATCH_SIZE) + ')':>14}  {'Δ':>8}")
print("-" * 70)
labels_proj = ["Self-play", "Training steps", "Eval vs MCTS"]
seq_vals = [t_selfplay_seq, t_training, t_eval]
bat_vals = [t_selfplay_bat, t_training, t_eval]
for label, sv, bv in zip(labels_proj, seq_vals, bat_vals):
    delta = sv - bv
    print(f"  {label:<28}  {sv:>10.1f}s  {bv:>12.1f}s  {'-' if delta >= 0 else '+'}{abs(delta):>6.1f}s")
print("-" * 70)
print(f"  {'TOTAL':<28}  {t_total_seq:>10.1f}s  {t_total_bat:>12.1f}s  ({t_total_seq / t_total_bat:.2f}× speedup)")
print(f"  {'':28}  {t_total_seq / 60:>9.1f}m  {t_total_bat / 60:>11.1f}m")

## 9. Summary charts

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("AlphaZero Connect4 — Time Analysis", fontsize=13, fontweight="bold")

# ── Chart 1: Iteration breakdown (sequential vs batched) ──────────────────
ax = axes[0, 0]
phase_labels_proj = ["Self-play", "Training", "Eval vs MCTS"]
seq_s = [t_selfplay_seq, t_training, t_eval]
bat_s = [t_selfplay_bat, t_training, t_eval]
x = np.arange(len(phase_labels_proj))
w = 0.35
b1 = ax.bar(x - w / 2, seq_s, w, label="Sequential run()", color="#e74c3c", edgecolor="white")
b2 = ax.bar(x + w / 2, bat_s, w, label=f"Batched batch={GAME_BATCH_SIZE}", color="#3498db", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(phase_labels_proj)
ax.set_ylabel("Time (s)")
ax.set_title("Time per iteration\n(sequential vs batched, single process)")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
for bar in [*b1, *b2]:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(max(seq_s), max(bat_s)) * 0.01,
        f"{bar.get_height():.0f}s",
        ha="center",
        va="bottom",
        fontsize=8,
    )

# ── Chart 2: MCTS phases ──────────────────────────────────────────────────
ax = axes[0, 1]
phase_labels_mcts = list(phase_times.keys())
phase_vals_mcts = [phase_times[k] for k in phase_labels_mcts]
phase_colors = ["#9b59b6", "#e67e22", "#e74c3c", "#1abc9c"]
wedges, texts, autotexts = ax.pie(
    phase_vals_mcts,
    labels=phase_labels_mcts,
    colors=phase_colors,
    autopct="%1.1f%%",
    startangle=90,
    pctdistance=0.75,
)
for at in autotexts:
    at.set_fontsize(9)
ax.set_title(f"MCTS phases\n({N_PROFILE_SIMS} simulations, sequential)")

# ── Chart 3: Micro-benchmarks ─────────────────────────────────────────────
ax = axes[1, 0]
mb_labels = list(micro.keys())
mb_vals_us = [micro[k] * 1e6 for k in mb_labels]
sorted_pairs = sorted(zip(mb_vals_us, mb_labels), reverse=True)
mb_vals_sorted, mb_labels_sorted = zip(*sorted_pairs)
bars = ax.barh(mb_labels_sorted, mb_vals_sorted, color="#34495e", edgecolor="white")
ax.set_xlabel("Time (µs)")
ax.set_title("Micro-benchmarks\n(µs per call)")
ax.set_xscale("log")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}"))
for bar, v in zip(bars, mb_vals_sorted):
    ax.text(v * 1.05, bar.get_y() + bar.get_height() / 2, f"{v:.1f}", va="center", fontsize=8)
ax.grid(True, alpha=0.3, axis="x")

# ── Chart 4: Batch speedup curve ──────────────────────────────────────────
ax = axes[1, 1]
bs_list_plot = list(batch_bench.keys())
speedups_plot = [batch_bench[bs]["speedup"] for bs in bs_list_plot]
ax.axhline(1.0, color="tomato", linewidth=2, linestyle="--", label="run()  baseline  (1×)")
ax.plot(bs_list_plot, speedups_plot, "o-", color="steelblue", linewidth=2, markersize=8, label="run_batched()")
ax.axvline(GAME_BATCH_SIZE, color="grey", linewidth=1.5, linestyle=":", label=f"config batch_size={GAME_BATCH_SIZE}")
for x_val, y_val in zip(bs_list_plot, speedups_plot):
    ax.annotate(f"{y_val:.2f}×", (x_val, y_val), textcoords="offset points", xytext=(5, 6), fontsize=9)
ax.set_xlabel("Batch size")
ax.set_ylabel("Speedup  (×)")
ax.set_title(f"Batched MCTS speedup\n({N_SIMS} simulations per search)")
ax.set_xticks(bs_list_plot)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
plt.savefig("time_analysis.png")